In [0]:
from pyspark.sql.functions import col, when, count, trim, initcap, to_date, to_timestamp

# 1. Cargar datos 
df = spark.table("default.ingesta_datosV2")

# 2. Diagnóstico de nulos por columna 
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).display()

# 3. Tipado correcto de columnas 
df = (
    df
    .withColumn("fecha_cit", to_date(col("fecha_cit"), "yyyy-MM-dd"))
    .withColumn("fechaNac_pac", to_date(col("fechaNac_pac"), "yyyy-MM-dd"))
    .withColumn("hr_inicio_cit", col("hr_inicio_cit").cast("string"))  
    .withColumn("hr_fin_cit", col("hr_fin_cit").cast("string"))
    .withColumn("mins_cit", col("mins_cit").cast("integer"))
    .withColumn("edad_pac_cita", col("edad_pac_cita").cast("integer"))
    .withColumn("peso_kg", col("peso_kg").cast("double"))
    .withColumn("altura_m", col("altura_m").cast("double"))
    .withColumn("imc", col("imc").cast("double"))
    .withColumn("pago_clie", col("pago_clie").cast("double"))
    .withColumn("pago_aseg", col("pago_aseg").cast("double"))
    .withColumn("pago_total", col("pago_total").cast("double"))
)

# --- 4. Manejo de nulos específico (no fillna global) ---
# nom_aseguradora nula = paciente sin seguro / pago particular
df = df.fillna({"nom_aseguradora": "Particular / Sin Seguro"})

# --- 5. Limpieza de texto (espacios extra, capitalización consistente) ---
df = (
    df
    .withColumn("nom_comp_pac", trim(initcap(col("nom_comp_pac"))))
    .withColumn("nom_comp_doc", trim(initcap(col("nom_comp_doc"))))
    .withColumn("motivo_cita", trim(col("motivo_cita")))
    .withColumn("especialidad_medica", trim(col("especialidad_medica")))
    .withColumn("nom_sucursal", trim(col("nom_sucursal")))
)

#6. Eliminar duplicados 
df = df.dropDuplicates()

# 7. Validación de anomalías (filas que no deberían existir) 
# Pagos negativos
df.filter((col("pago_clie") < 0) | (col("pago_aseg") < 0) | (col("pago_total") < 0)).display()

# Consistencia: pago_total debe ser igual a pago_clie + pago_aseg
df.filter(col("pago_total") != (col("pago_clie") + col("pago_aseg"))).display()

# Edad fuera de rango razonable (0-110 años)
df.filter((col("edad_pac_cita") < 0) | (col("edad_pac_cita") > 110)).display()

# Duración de cita fuera de valores esperados (15/30/45/60 min)
df.filter(~col("mins_cit").isin(15, 30, 45, 60)).display()

# 8. Verificación final de nulos (debe quedar en 0 en todas las columnas) 
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).display()

# 9. Guardar tabla limpia en el catálogo para el resto del equipo 
df.write.mode("overwrite").saveAsTable("workspace.default.citas_pmm_limpioV2")

print(f"Filas finales: {df.count()}")


fecha_cit,hr_inicio_cit,hr_fin_cit,mins_cit,motivo_cita,estado_cita,nom_comp_pac,sexo_pac,fechaNac_pac,edad_pac_cita,peso_kg,altura_m,imc,tipo_sangre,nom_comp_doc,especialidad_medica,nom_sucursal,nom_aseguradora,pago_clie,pago_aseg,pago_total
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2442,0,0,0


fecha_cit,hr_inicio_cit,hr_fin_cit,mins_cit,motivo_cita,estado_cita,nom_comp_pac,sexo_pac,fechaNac_pac,edad_pac_cita,peso_kg,altura_m,imc,tipo_sangre,nom_comp_doc,especialidad_medica,nom_sucursal,nom_aseguradora,pago_clie,pago_aseg,pago_total


fecha_cit,hr_inicio_cit,hr_fin_cit,mins_cit,motivo_cita,estado_cita,nom_comp_pac,sexo_pac,fechaNac_pac,edad_pac_cita,peso_kg,altura_m,imc,tipo_sangre,nom_comp_doc,especialidad_medica,nom_sucursal,nom_aseguradora,pago_clie,pago_aseg,pago_total


fecha_cit,hr_inicio_cit,hr_fin_cit,mins_cit,motivo_cita,estado_cita,nom_comp_pac,sexo_pac,fechaNac_pac,edad_pac_cita,peso_kg,altura_m,imc,tipo_sangre,nom_comp_doc,especialidad_medica,nom_sucursal,nom_aseguradora,pago_clie,pago_aseg,pago_total


fecha_cit,hr_inicio_cit,hr_fin_cit,mins_cit,motivo_cita,estado_cita,nom_comp_pac,sexo_pac,fechaNac_pac,edad_pac_cita,peso_kg,altura_m,imc,tipo_sangre,nom_comp_doc,especialidad_medica,nom_sucursal,nom_aseguradora,pago_clie,pago_aseg,pago_total


fecha_cit,hr_inicio_cit,hr_fin_cit,mins_cit,motivo_cita,estado_cita,nom_comp_pac,sexo_pac,fechaNac_pac,edad_pac_cita,peso_kg,altura_m,imc,tipo_sangre,nom_comp_doc,especialidad_medica,nom_sucursal,nom_aseguradora,pago_clie,pago_aseg,pago_total
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


Filas finales: 10000
